# Bloque 2 Tema 3: Ensambles, Stacking y Modelos Híbridos
## Aplicaciones Reales en Banca

En este notebook exploraremos tres estrategias de ensambles aplicadas a casos reales de banca:
1. **Voting Ensemble**: Combining predictions from multiple models
2. **Stacking**: Training a meta-model to learn optimal weights
3. **Evaluación comparativa**: Cuándo cada estrategia es más efectiva

**Filosofía**: No grafos complejos. Solo lo necesario para entender el impacto de cada técnica.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression  # Corregido: estava en sklearn.ensemble
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix
from sklearn.datasets import make_classification
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Colab: usa 'default' si 'seaborn-v0_8-darkgrid' no está disponible
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    plt.style.use('default')

---
## CASO 1: DETECCIÓN DE FRAUDE CON VOTING ENSEMBLE

**Contexto**: Banco quiere combinar tres modelos de especialidades diferentes (árbol, gradiente, regresión) para detectar transacciones fraudulentas. 

**Pregunta clave**: ¿El promedio simple de predicciones mejora la generalización?

In [ ]:
# Generar dataset de fraude: 1000 transacciones, 50 features
# Clase desequilibrada (típico en fraude): 5% fraude, 95% legítimo
X_fraud, y_fraud = make_classification(
    n_samples=1000, n_features=50, n_informative=25, n_redundant=10,
    weights=[0.95, 0.05], random_state=42, flip_y=0.02
)

# Split: 60% train, 20% validation, 20% test
X_train_f, X_temp_f, y_train_f, y_temp_f = train_test_split(
    X_fraud, y_fraud, test_size=0.4, random_state=42, stratify=y_fraud
)
X_val_f, X_test_f, y_val_f, y_test_f = train_test_split(
    X_temp_f, y_temp_f, test_size=0.5, random_state=42, stratify=y_temp_f
)

print(f"Train: {X_train_f.shape[0]} | Val: {X_val_f.shape[0]} | Test: {X_test_f.shape[0]}")
print(f"Fraude en train: {y_train_f.mean():.2%}")

In [ ]:
# Entrenar 3 modelos base (especialidades diferentes)
model_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
model_gb = GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42)
model_lr = LogisticRegression(max_iter=1000, random_state=42)

# Escalar para regresión logística
scaler = StandardScaler()
X_train_f_scaled = scaler.fit_transform(X_train_f)
X_val_f_scaled = scaler.transform(X_val_f)
X_test_f_scaled = scaler.transform(X_test_f)

# Entrenar
model_rf.fit(X_train_f, y_train_f)
model_gb.fit(X_train_f, y_train_f)
model_lr.fit(X_train_f_scaled, y_train_f)

# Predicciones en test
pred_rf_test = model_rf.predict_proba(X_test_f)[:, 1]
pred_gb_test = model_gb.predict_proba(X_test_f)[:, 1]
pred_lr_test = model_lr.predict_proba(X_test_f_scaled)[:, 1]

# Voting: promedio simple
pred_voting = (pred_rf_test + pred_gb_test + pred_lr_test) / 3

# Evaluar AUC
auc_rf = roc_auc_score(y_test_f, pred_rf_test)
auc_gb = roc_auc_score(y_test_f, pred_gb_test)
auc_lr = roc_auc_score(y_test_f, pred_lr_test)
auc_voting = roc_auc_score(y_test_f, pred_voting)

print(f"AUC Random Forest:  {auc_rf:.4f}")
print(f"AUC Gradient Boost: {auc_gb:.4f}")
print(f"AUC Logistic Reg:   {auc_lr:.4f}")
print(f"\n✅ AUC Voting Ensemble: {auc_voting:.4f}")

In [ ]:
# Visualizar: AUC de modelos individuales vs ensemble
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: AUC Comparison
models = ['Random\nForest', 'Gradient\nBoost', 'Logistic\nReg', 'Voting\nEnsemble']
aucs = [auc_rf, auc_gb, auc_lr, auc_voting]
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

bars = ax1.bar(models, aucs, color=colors, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax1.set_ylim([0.75, 0.95])
ax1.set_title('Caso 1: Detección de Fraude - Comparación AUC', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Agregar valores en barras
for bar, auc in zip(bars, aucs):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{auc:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Panel 2: Mejora relativa del ensemble
improvements = [(auc_voting - auc) / auc * 100 for auc in [auc_rf, auc_gb, auc_lr]]
model_pairs = ['vs RF', 'vs GB', 'vs LR']

bars2 = ax2.bar(model_pairs, improvements, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Mejora Relativa (%)', fontsize=11, fontweight='bold')
ax2.set_title('Ganancia del Voting Ensemble', fontsize=12, fontweight='bold')
ax2.axhline(y=0, color='black', linestyle='--', linewidth=0.8)
ax2.grid(axis='y', alpha=0.3)

for bar, imp in zip(bars2, improvements):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{imp:.1f}%', ha='center', va='bottom' if height > 0 else 'top', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('caso1_voting_fraud.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nMejoras relativas del Ensemble:")
for pair, imp in zip(model_pairs, improvements):
    print(f"  {pair}: {imp:+.2f}%")

### ✅ CONCLUSIÓN CASO 1: VOTING ENSEMBLE

**Hallazgo**: El voting ensemble alcanza un AUC de **{:.4f}**, mejorando el mejor modelo individual (GB: {:.4f}) en **{:.2f}%**.

**Por qué funciona**: Cada modelo captura diferentes patrones en las transacciones fraudulentas:
- Random Forest: Relaciones no-lineales complejas
- Gradient Boosting: Interacciones secuenciales
- Logistic Regression: Relaciones lineales robustas

**Ventaja principal**: Robustez. Si un modelo falla en cierto subconjunto de datos, otros compensan.

**Regla de oro**: Usa Voting cuando tus modelos base tienen especialidades complementarias y bajo sesgo individual.\n".format(auc_voting, auc_gb, (auc_voting - auc_gb) / auc_gb * 100)

---
## CASO 2: SCORING DE CRÉDITO CON STACKING

**Contexto**: Banco otorga créditos usando scores de riesgo. Necesita que los rankings de clientes por riesgo sean estables y bien calibrados.

**Pregunta clave**: ¿El stacking mejora la calibración y consistencia del scoring?

In [ ]:
# Dataset de scoring: 800 clientes, 40 features (ingresos, historial, edad, etc.)
X_credit, y_credit = make_classification(
    n_samples=800, n_features=40, n_informative=20, n_redundant=8,
    weights=[0.7, 0.3], random_state=42
)

# Split estratificado para train/val/test (importante para stacking)
X_train_c, X_temp_c, y_train_c, y_temp_c = train_test_split(
    X_credit, y_credit, test_size=0.35, random_state=42, stratify=y_credit
)
X_val_c, X_test_c, y_val_c, y_test_c = train_test_split(
    X_temp_c, y_temp_c, test_size=0.5, random_state=42, stratify=y_temp_c
)

print(f"Train: {X_train_c.shape[0]} | Val: {X_val_c.shape[0]} | Test: {X_test_c.shape[0]}")

In [ ]:
# STEP 1: Entrenar modelos base en datos de train
base_rf = RandomForestClassifier(n_estimators=50, max_depth=8, random_state=42)
base_gb = GradientBoostingClassifier(n_estimators=50, max_depth=4, random_state=42)
base_lr = LogisticRegression(max_iter=500, random_state=42)

base_rf.fit(X_train_c, y_train_c)
base_gb.fit(X_train_c, y_train_c)
scaler_c = StandardScaler()
X_train_c_scaled = scaler_c.fit_transform(X_train_c)
base_lr.fit(X_train_c_scaled, y_train_c)

print("✓ Modelos base entrenados")

# STEP 2: Generar features para meta-model usando datos de validación
# Crucial: usar datos diferentes de los que entrenamos los modelos base
X_val_c_scaled = scaler_c.transform(X_val_c)

meta_features_val = np.column_stack([
    base_rf.predict_proba(X_val_c)[:, 1],
    base_gb.predict_proba(X_val_c)[:, 1],
    base_lr.predict_proba(X_val_c_scaled)[:, 1]
])

print(f"✓ Meta-features generadas: {meta_features_val.shape}")

# STEP 3: Entrenar meta-model (regresión logística = ajustar pesos)
meta_model = LogisticRegression(max_iter=100, random_state=42)
meta_model.fit(meta_features_val, y_val_c)

# Obtener pesos del meta-model (cómo pondera cada modelo base)
meta_weights = meta_model.coef_[0]
meta_intercept = meta_model.intercept_[0]

print(f"\n✓ Meta-model entrenado")
print(f"  Pesos aprendidos (RF, GB, LR): {meta_weights}")
print(f"  Intercepto: {meta_intercept:.4f}")

In [ ]:
# STEP 4: Aplicar stacking en datos de test
X_test_c_scaled = scaler_c.transform(X_test_c)

pred_rf_test_c = base_rf.predict_proba(X_test_c)[:, 1]
pred_gb_test_c = base_gb.predict_proba(X_test_c)[:, 1]
pred_lr_test_c = base_lr.predict_proba(X_test_c_scaled)[:, 1]

# Stacking: predicción ponderada
meta_features_test = np.column_stack([pred_rf_test_c, pred_gb_test_c, pred_lr_test_c])
pred_stacking = meta_model.predict_proba(meta_features_test)[:, 1]

# Voting simple para comparación
pred_voting_c = (pred_rf_test_c + pred_gb_test_c + pred_lr_test_c) / 3

# AUC
auc_rf_c = roc_auc_score(y_test_c, pred_rf_test_c)
auc_gb_c = roc_auc_score(y_test_c, pred_gb_test_c)
auc_voting_c = roc_auc_score(y_test_c, pred_voting_c)
auc_stacking = roc_auc_score(y_test_c, pred_stacking)

print(f"AUC Random Forest:      {auc_rf_c:.4f}")
print(f"AUC Gradient Boost:     {auc_gb_c:.4f}")
print(f"AUC Voting:             {auc_voting_c:.4f}")
print(f"\n✅ AUC Stacking:        {auc_stacking:.4f}")

In [ ]:
# Visualizar: Stacking vs alternativas
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: AUC comparison
models_c = ['RF', 'GB', 'Voting', 'Stacking']
aucs_c = [auc_rf_c, auc_gb_c, auc_voting_c, auc_stacking]
colors_c = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

bars = ax1.bar(models_c, aucs_c, color=colors_c, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('AUC-ROC', fontsize=11, fontweight='bold')
ax1.set_ylim([0.70, 0.88])
ax1.set_title('Caso 2: Scoring de Crédito - AUC', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for bar, auc in zip(bars, aucs_c):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{auc:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Panel 2: Pesos del meta-model (importancia de cada modelo base)
base_names = ['Random\nForest', 'Gradient\nBoost', 'Logistic\nReg']
normalized_weights = meta_weights / meta_weights.sum()

bars2 = ax2.bar(base_names, normalized_weights, color=['#1f77b4', '#ff7f0e', '#2ca02c'], alpha=0.7, edgecolor='black', linewidth=1.5)
ax2.set_ylabel('Peso Normalizado', fontsize=11, fontweight='bold')
ax2.set_title('Pesos Aprendidos por Meta-Model', fontsize=12, fontweight='bold')
ax2.set_ylim([0, 0.6])
ax2.grid(axis='y', alpha=0.3)

for bar, w in zip(bars2, normalized_weights):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height,
             f'{w:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('caso2_stacking_credit.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"\nPesos normalizados de meta-model:")
for name, w in zip(base_names, normalized_weights):
    print(f"  {name.replace(chr(10), ' ')}: {w:.1%}")

### ✅ CONCLUSIÓN CASO 2: STACKING

**Hallazgo**: El stacking alcanza AUC de **{:.4f}**, mejorando voting en **{:.2f}%**.

**Por qué es mejor**: Stacking NO usa promedio simple. Aprende pesos óptimos:
- Random Forest: {:.1f}% (fuerte en correlaciones)
- Gradient Boost: {:.1f}% (fuerte en no-linealidades)
- Logistic Reg: {:.1f}% (valida líneas de decisión)

**Ventaja principal**: Adaptabilidad. El meta-model ajusta automáticamente qué especialidad es más importante en tus datos específicos.

**Regla de oro**: Usa Stacking cuando tienes datos suficientes (800+ muestras) y modelos base entrenados bien. Requiere validación cuidadosa para evitar leakage.\n".format(auc_stacking, (auc_stacking - auc_voting_c) / auc_voting_c * 100, normalized_weights[0]*100, normalized_weights[1]*100, normalized_weights[2]*100)

---
## CASO 3: PREDICCIÓN DE DEFAULT CON HÍBRIDO

**Contexto**: Banco predice qué clientes harán default en los próximos 12 meses. Usa historial de 3 años. Problema: datos temporales → riesgo de leakage.

**Pregunta clave**: ¿Cómo validar ensambles sin caer en leakage temporal?

In [ ]:
# Dataset temporal: 1200 clientes, 30 features, 40% default (alto riesgo)
X_default, y_default = make_classification(
    n_samples=1200, n_features=30, n_informative=15, n_redundant=6,
    weights=[0.6, 0.4], random_state=42, flip_y=0.05
)

# Simular estructura temporal: índices son clientes ordenados por tiempo
# Primera 60% de datos = historial pasado
# Última 40% = período de predicción (2024-2025)

split_temporal = int(len(X_default) * 0.6)

X_historical = X_default[:split_temporal]
y_historical = y_default[:split_temporal]
X_future = X_default[split_temporal:]
y_future = y_default[split_temporal:]

# Para stacking, dividir histórico en train y calibración
X_train_d, X_calib_d, y_train_d, y_calib_d = train_test_split(
    X_historical, y_historical, test_size=0.3, random_state=42, stratify=y_historical
)

print(f"Historial (2021-2023): {X_historical.shape[0]} | "
      f"Train: {X_train_d.shape[0]} | Calibración: {X_calib_d.shape[0]}")
print(f"Futuro (2024-2025):    {X_future.shape[0]}")
print(f"Proporción default en futuro: {y_future.mean():.1%}")

In [ ]:
# STEP 1: Entrenar 2 modelos base en histórico
model_rf_d = RandomForestClassifier(n_estimators=80, max_depth=10, random_state=42)
model_gb_d = GradientBoostingClassifier(n_estimators=80, max_depth=5, random_state=42)

model_rf_d.fit(X_train_d, y_train_d)
model_gb_d.fit(X_train_d, y_train_d)

print("✓ Modelos base entrenados en histórico")

# STEP 2: Calibrar meta-model en datos NO vistos por modelos base
meta_features_calib = np.column_stack([
    model_rf_d.predict_proba(X_calib_d)[:, 1],
    model_gb_d.predict_proba(X_calib_d)[:, 1]
])

meta_model_d = LogisticRegression(max_iter=100, random_state=42)
meta_model_d.fit(meta_features_calib, y_calib_d)

print("✓ Meta-model calibrado en datos de validación")

# STEP 3: Predecir en período futuro (verdadera prueba temporal)
pred_rf_future = model_rf_d.predict_proba(X_future)[:, 1]
pred_gb_future = model_gb_d.predict_proba(X_future)[:, 1]

# Stacking
meta_features_future = np.column_stack([pred_rf_future, pred_gb_future])
pred_stacking_d = meta_model_d.predict_proba(meta_features_future)[:, 1]

# Voting
pred_voting_d = (pred_rf_future + pred_gb_future) / 2

# AUC en futuro
auc_rf_d = roc_auc_score(y_future, pred_rf_future)
auc_gb_d = roc_auc_score(y_future, pred_gb_future)
auc_voting_d = roc_auc_score(y_future, pred_voting_d)
auc_stacking_d = roc_auc_score(y_future, pred_stacking_d)

print(f"\nAUC en período futuro (2024-2025):")
print(f"  Random Forest:       {auc_rf_d:.4f}")
print(f"  Gradient Boost:      {auc_gb_d:.4f}")
print(f"  Voting:              {auc_voting_d:.4f}")
print(f"  ✅ Stacking Híbrido: {auc_stacking_d:.4f}")

In [ ]:
# Visualizar: Robustez temporal del stacking
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Panel 1: AUC comparison
models_d = ['RF', 'GB', 'Voting', 'Stacking']
aucs_d = [auc_rf_d, auc_gb_d, auc_voting_d, auc_stacking_d]
colors_d = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

bars = ax1.bar(models_d, aucs_d, color=colors_d, alpha=0.7, edgecolor='black', linewidth=1.5)
ax1.set_ylabel('AUC-ROC (Período Futuro)', fontsize=11, fontweight='bold')
ax1.set_ylim([0.65, 0.82])
ax1.set_title('Caso 3: Predicción de Default - Validación Temporal', fontsize=12, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

for bar, auc in zip(bars, aucs_d):
    height = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2., height,
             f'{auc:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Panel 2: Distribución de scores en futuro
ax2.hist(pred_stacking_d[y_future == 0], bins=20, alpha=0.6, label='No Default (n={})'.format((y_future == 0).sum()), color='#2ca02c')
ax2.hist(pred_stacking_d[y_future == 1], bins=20, alpha=0.6, label='Default (n={})'.format((y_future == 1).sum()), color='#d62728')
ax2.set_xlabel('Score de Stacking', fontsize=11, fontweight='bold')
ax2.set_ylabel('Frecuencia', fontsize=11, fontweight='bold')
ax2.set_title('Distribución de Scores - Separación de Clases', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=10)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('caso3_stacking_default.png', dpi=100, bbox_inches='tight')
plt.show()

### ✅ CONCLUSIÓN CASO 3: HÍBRIDO CON VALIDACIÓN TEMPORAL

**Hallazgo**: El stacking híbrido mantiene AUC de **{:.4f}** en período futuro (mejor que RF: {:.4f}).

**Estructura de validación (crítica para evitar leakage)**:
1. Entrenar modelos base en histórico (2021-2023)
2. Calibrar meta-model en validación diferente del histórico
3. Evaluar en futuro (2024-2025) sin nunca exponer esos datos

**Por qué esto importa**: En banca, un modelo que ve el futuro es inútil. La validación temporal garantiza que predicciones generalizan a clientes que aún no ocurren.

**Regla de oro**: En datos temporales, SIEMPRE:
- Train ← histórico
- Meta-calibration ← validación diferente
- Test ← futuro

Nunca mezcles estos períodos.\n".format(auc_stacking_d, auc_rf_d)

---
## TABLA DE DECISIÓN: CUÁNDO USAR CADA ESTRATEGIA

| Contexto | Datos | Modelos Base | Recomendación | Razón |
|---|---|---|---|---|
| **Rápido y simple** | <1000 muestras | Cualquiera | **Voting** | Bajo overhead, interpretable |
| **Datos abundantes** | >1000 muestras | Varios tipos | **Stacking** | Aprende pesos óptimos, mejor AUC |
| **Especialidades claras** | Cualquiera | Muy diferentes | **Voting** | Cada modelo trae perspectiva única |
| **Series temporales** | Histórico + futuro | Cualquiera | **Stacking + Split Temporal** | Evita leakage, valida robustez |
| **Alta desconfianza** | Desbalanceado | Cualquiera | **Voting** | Menos overfitting a ruido |
| **Producción exigente** | >100K ejemplos | Varios tipos | **Stacking** | Máxima performance, explota correlaciones |

### Reglas de Oro

1. **Diversidad > Cantidad**: 3 modelos buenos y diferentes > 10 árboles similares
2. **Leakage es el enemigo**: Si tu meta-model ve datos de test, no confíes en métricas
3. **Stacking requiere escala**: <500 muestras → Voting. >500 → Stacking es viable
4. **Simplicidad gana**: Si Voting y Stacking dan AUC similar, elige Voting (mantenible)

---
## RESUMEN EJECUTIVO

### Hallazgos Clave

**1. Voting Ensemble (Caso Fraude)**
- Combina 3 modelos con promedio simple → AUC {:.4f}
- Mejora el mejor modelo individual en ~{:.1f}%
- Ventaja: Robustez, fácil de implementar
- Riesgo: Puede suavizar decisiones importantes

**2. Stacking (Caso Scoring de Crédito)**
- Entrena meta-model que aprende pesos óptimos
- AUC {:.4f} vs Voting {:.4f} → mejora de {:.1f}%
- Ventaja: Máxima performance teórica
- Riesgo: Requiere validación cuidadosa, puede overfitear

**3. Validación Temporal (Caso Default)**
- Estructura: Train histórico → Calibración → Test futuro
- Evita leakage, garantiza generalización real
- Stacking temporal mantiene AUC {:.4f} en período futuro
- Lección: En banca, la validación temporal es no-negociable

### Decisiones Prácticas para tu Banco

**Escenario 1: Detección de Fraude** → Usa Voting
- Razón: Necesitas decisiones rápidas (real-time), diversidad de patrones
- Datos: >500K transacciones diarias
- Latencia: <100ms → Voting es más rápido que Stacking

**Escenario 2: Scoring de Crédito** → Usa Stacking
- Razón: Decisiones offline, datos suficientes (800+ aprobaciones/mes)
- Datos: Histórico consolidado de 5 años
- Performance: AUC importa más que latencia

**Escenario 3: Predicción de Default** → Usa Stacking + Validación Temporal
- Razón: Datos series temporales, riesgo regulatorio si overfiteas
- Datos: Histórico rolling de 3 años + validación forward
- Governance: Auditabilidad completa de splits temporales

### Implementación Next Steps

1. Verifica que tus modelos base sean realmente diferentes (cor < 0.6)
2. Valida sin leakage: train/val/test sin solapamientos
3. En producción: monitorea AUC mensualmente, recalibra meta-model si cae >5%
4. Documenta decisión: cuál ensemble eligiste y por qué
".format(auc_voting, (auc_voting-auc_gb)/auc_gb*100, auc_stacking, auc_voting_c, (auc_stacking-auc_voting_c)/auc_voting_c*100, auc_stacking_d)